# **Category Database Creation and Linking**
This notebook categorizes meeting items and creates category nodes in the Neo4j database.
The categories are assigned by an LLM based on meeting item context and linked to the knowledge graph.

## **1. Load Environment and Imports**

In [7]:
import os
import json
import time
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Load environment variables
load_dotenv("../config/config.env")
load_dotenv("../config/secrets.env")

# Import category linker functions
from data_pipeline.category_linker import (
    fetch_meeting_items_from_db,
    create_category_batch_file,
    submit_category_batch,
    check_category_batch_status,
    retrieve_batch_output,
    parse_category_results,
    create_category_nodes,
    link_meeting_items_to_categories,
    get_categorization_stats
)

## **2. Connect to Neo4j Database**

In [2]:
# Connect to Neo4j
uri = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(username, password))

# Test connection
with driver.session() as session:
    result = session.run("RETURN 1")
    print("Connected to Neo4j successfully!")
    print(f"Neo4j URI: {uri}")

Connected to Neo4j successfully!
Neo4j URI: neo4j://127.0.0.1:7687


## **3. Load Categories and Meeting Items**

In [13]:
# Load categories.json
categories_path = os.path.join(os.getenv("PROTOCOLS_PATH"), "categories.json")
with open(categories_path, "r", encoding="utf-8") as f:
    categories_data = json.load(f)

print(f"Loaded categories from {categories_path}")
print(f"Total categories: {len(categories_data.get('categories', []))}")

# Fetch all meeting items from database
meeting_items = fetch_meeting_items_from_db(driver)

Loaded categories from ../data/protocols/categories.json
Total categories: 3
Fetched 100 meeting items from database


## **4. Create Category Batch File**

In [10]:
# Load the categorization prompt
prompt_path = os.getenv("CATEGORY_EXTRACTION_PROMPT_PATH")
with open(prompt_path, "r", encoding="utf-8") as f:
    prompt = f.read()

print(f"Loaded prompt from {prompt_path}")

# Create batch file for categorization
batch_file_path = os.getenv("CATEGORY_BATCH_FILE_PATH")
create_category_batch_file(meeting_items, categories_data, prompt, batch_file_path)

Loaded prompt from ../data/llm/prompts/category_extraction_prompt.txt
Overwriting batch file...


Creating batch tasks: 100%|███████████████████████████████████████████████| 100/100 [00:00<00:00, 3099.93it/s]

Batch file created at ../data/temp/category_batch.jsonl with 100 tasks


## **5. Submit Batch Job to OpenAI**

In [11]:
# Submit batch job
batch_id = submit_category_batch(
    batch_file_path,
    os.getenv("CATEGORY_BATCH_INPUT_ID_SAVE_PATH")
)

print(f"\nBatch submitted with ID: {batch_id}")
print("You can check the status in the next cell while waiting for completion.")

Batch job submitted successfully
Batch ID: batch_69aa8bb6a454819086af89f8e9c93aa2
Batch ID saved at: ../data/temp/category_batch_file_id.txt

Batch submitted with ID: batch_69aa8bb6a454819086af89f8e9c93aa2
You can check the status in the next cell while waiting for completion.


## **6. Poll Batch Status**
> This may take a few minutes to hours depending on the number of meeting items.
> You can run this cell periodically to check the status.

In [12]:
import time

# If you have a saved batch ID, load it
# batch_id = "batch_..."  # Replace with your batch ID

print(f"Checking status for batch: {batch_id}")
output_id = None

while output_id is None:
    output_id = check_category_batch_status(batch_id)
    if output_id is None:
        print("Batch still processing... waiting 10 seconds before next check")
        time.sleep(10)

print(f"\nBatch completed! Output file ID: {output_id}")

Checking status for batch: batch_69aa8bb6a454819086af89f8e9c93aa2
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before next check
Batch still processing... waiting 10 seconds before 

KeyboardInterrupt: 

## **7. Retrieve and Parse Results**

In [ ]:
# Retrieve batch output from OpenAI
output_jsonl = retrieve_batch_output(output_id)

if output_jsonl is None:
    print("Error retrieving batch output!")
else:
    print(f"Retrieved output with {len(output_jsonl.splitlines())} lines")

# Parse category results
category_results = parse_category_results(output_jsonl)

print(f"\nParsing complete!")
print(f"Successfully categorized items: {len(category_results)}")

# Show sample results
sample_count = 0
for item_id, categories in category_results.items():
    if sample_count < 3:
        print(f"\nItem {sample_count + 1}:")
        print(f"  Item ID: {item_id}")
        print(f"  Categories: {len(categories)}")
        for cat in categories[:2]:
            print(f"    - {cat.get('category', 'N/A')} > {cat.get('subcategory', 'N/A')} (confidence: {cat.get('confidence', 0.0):.2f})")
        sample_count += 1
    else:
        break

## **8. Create Category Nodes in Database**

In [ ]:
# Create category nodes in Neo4j
create_category_nodes(driver, categories_data)

print("\nCategory nodes created successfully!")

## **9. Link Meeting Items to Categories**

In [ ]:
# Link meeting items to categories
link_meeting_items_to_categories(driver, category_results)

print("\nMeeting items linked to categories successfully!")

## **10. View Categorization Statistics**

In [ ]:
# Get and display statistics
stats = get_categorization_stats(driver)

print("\n=== Categorization Statistics ===")
print(f"Total Category nodes: {stats['total_categories']}")
print(f"Total Subcategory nodes: {stats['total_subcategories']}")
print(f"Total SubSubcategory nodes: {stats['total_sub_subcategories']}")
print(f"\nMeeting items with categories: {stats['items_with_categories']}")
print(f"Total category links: {stats['total_category_links']}")
print(f"Average categories per item: {stats['total_category_links'] / max(stats['items_with_categories'], 1):.2f}")

## **11. Test Category Queries**
> Try some Cypher queries to verify the category structure and links

In [ ]:
# Example query: Get all meeting items in a specific category
category_name = "Barn och skola"

with driver.session() as session:
    result = session.run("""
        MATCH (c:Category {name: $category_name})-[:HAS_SUBCATEGORY|HAS_SUB_SUBCATEGORY*0..]-(cat)
        <-[:HAS_CATEGORY]-(mi:MeetingItem)
        RETURN DISTINCT mi.title as title, count(*) as match_count
        LIMIT 10
    """, category_name=category_name)
    
    items = [record for record in result]
    print(f"\nMeeting items in '{category_name}':")
    for i, record in enumerate(items, 1):
        print(f"{i}. {record['title']}")
    
    if not items:
        print("No items found in this category")

In [ ]:
# Example query: Find most common categories
with driver.session() as session:
    result = session.run("""
        MATCH (mi:MeetingItem)-[r:HAS_CATEGORY]->(cat)
        WHERE cat.name IS NOT NULL
        RETURN cat.name as category, avg(r.confidence) as avg_confidence, count(*) as count
        ORDER BY count DESC
        LIMIT 15
    """)
    
    categories = [record for record in result]
    print("\nMost common categories assigned:")
    for i, record in enumerate(categories, 1):
        print(f"{i}. {record['category']}: {record['count']} items (avg confidence: {record['avg_confidence']:.2f})")

## **12. Close Database Connection**

In [ ]:
# Close Neo4j connection
driver.close()
print("Database connection closed.")